# MHT-DataHub — full pipeline (S0 → S10 + dashboard)

Turns a folder of PDF papers on multiphase heat transfer into an auto-labeled,
evidence-grounded catalog, digitizes the numeric data buried in each paper's
figures, and builds the self-contained offline dashboard (`app/dist/index.html`).

| Stage | What it does | Model? | Code |
|---|---|---|---|
| S0 ingest | PDF → text / pages / sections / figure crops | no | `mhtdb/s0_ingest.py` |
| S1 triage | paper type, dataset gate | LLM | `mhtdb/extract.py` |
| S2 conditions | numeric operating envelope + evidence quotes | LLM | `mhtdb/extract.py` |
| S3 taxonomy | facets from the controlled vocabulary | LLM | `mhtdb/extract.py` |
| S4 application | inferred application target | LLM | `mhtdb/extract.py` |
| S5 normalize | units → SI, CoolProp, dimensionless groups | no | `mhtdb/normalize.py` |
| S6 verify | evidence check (+ optional LLM contradiction audit) | partial | `mhtdb/pipeline.py` + `mhtdb/extract.py` |
| S7 commit | write `catalog/records/<id>.json` | no | `mhtdb/pipeline.py` |
| S8 crops | crop every figure/table into its own PDF + PNG | no | `mhtdb/figure_crops.py` |
| S9 digitize | recover (x, y) data points from the cropped figures | LLM (agentic) | `mhtdb/digitize.py` |
| S10 curves | compile points → CSV + boiling-curve comparison plot | no | `mhtdb/curves.py` |
| build | compile `catalog/` → the standalone dashboard HTML | no | `app/build.py` |

Governing rule (see `PLAN.md`): **the LLM extracts and cites, Python computes
and classifies.** Every quote the model returns is checked against the source
PDF text; nothing derived (units, tags, dimensionless numbers) is guessed by
a model. S9 is the one deliberate exception: reading numbers off a figure
needs a model with vision and tool access, so it runs agentically
(`--tools` enabled) rather than as tools-disabled structured output like
S1–S4.

Run this notebook's cells top to bottom from a kernel whose working directory
is the project root (the folder this notebook lives in).

## Setup

In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    # Notebook was launched from somewhere else; assume it still lives at the
    # project root and fix the working directory rather than guess a path.
    ROOT = Path(__file__).resolve().parent if "__file__" in dir() else ROOT
    os.chdir(ROOT)
assert (ROOT / "pyproject.toml").exists(), f"not a project root: {ROOT}"

PY = sys.executable
print("project root:", ROOT)
print("python:", PY)


def run(args, check=True):
    """Run a CLI stage, streaming its own stdout/stderr straight to the cell.

    Piping and re-printing line by line (rather than letting the child
    inherit this process's stdout handle) matters on Windows: a Jupyter
    kernel's real OS-level stdout is not the same object ipykernel captures
    into the cell, so a child process's own prints can vanish from the
    notebook entirely -- including its error diagnostics -- if it's simply
    allowed to inherit that handle. PYTHONIOENCODING/PYTHONUTF8 keep the
    child's own encoding predictable regardless of the console code page.
    """
    print("$", " ".join(str(a) for a in args))
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONIOENCODING": "utf-8", "PYTHONUTF8": "1"}
    proc = subprocess.Popen(
        args, cwd=ROOT, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace", bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if check and proc.returncode != 0:
        raise SystemExit(f"command failed ({proc.returncode}): {' '.join(str(a) for a in args)}")
    return proc

project root: D:\a_claude\AgenticTool-MultiPhaseHUB-main - test11\AgenticTool-MultiPhaseHUB-main
python: C:\Users\DELL\AppData\Local\Programs\Python\Python312\python.exe


## Step 0 — Install dependencies

Core deps (`pymupdf`, `pydantic`, `anthropic`, `CoolProp`, `numpy`) plus the
`dev`/`plot` extras (`pytest`, `matplotlib`). S9 digitization reads figures
directly through an agentic `claude` CLI call rather than a separate OCR
pipeline, so no OCR engine is needed.

In [2]:
run([PY, "-m", "pip", "install", "-e", ".[dev,plot]"])

$ C:\Users\DELL\AppData\Local\Programs\Python\Python312\python.exe -m pip install -e .[dev,plot]


Obtaining file:///D:/a_claude/AgenticTool-MultiPhaseHUB-main%20-%20test11/AgenticTool-MultiPhaseHUB-main
  Installing build dependencies: started


  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started


  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started


  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started


...

  Building editable for mht-datahub (pyproject.toml): finished with status 'done'
  Created wheel for mht-datahub: filename=mht_datahub-0.1.0-0.editable-py3-none-any.whl size=4047 sha256=2a3bb979b1eec8ac4376c82347bc1672ec1392199b85b397005e321770594155
  Stored in directory: C:\Users\DELL\AppData\Local\Temp\pip-ephem-wheel-cache-fv2n98h2\wheels\6c\88\2e\eb9e01fcdaf62b9a850a0bf7c80f6616a8887b9ac013653dfe
Successfully built mht-datahub


  Attempting uninstall: mht-datahub
    Found existing installation: mht-datahub 0.1.0
    Uninstalling mht-datahub-0.1.0:


      Successfully uninstalled mht-datahub-0.1.0



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


<Popen: returncode: 0 args: ['C:\\Users\\DELL\\AppData\\Local\\Programs\\Pyt...>

### API / CLI backend setup

Which model backend S1–S4 extraction uses is decided right here, so it's
visible and overridable before any PDF is touched. Three backends satisfy
the same contract (`mhtdb/backends.py`):

| Backend | Needs | How it's driven |
|---|---|---|
| `api` | `ANTHROPIC_API_KEY` or `ANTHROPIC_AUTH_TOKEN` | `anthropic` Python SDK |
| `claude-code` | the `claude` CLI installed and logged in | subprocess, `--tools ""` (no tool use) |
| `codex` | the `codex` CLI installed and logged in | subprocess, `codex exec --output-schema` |

Auto-selection order (same as `mhtdb.backends.detect_backend()`): an
`MHTDB_BACKEND` env var override, else API credentials, else a local Claude
Code install, else a signed-in Codex CLI. Set `BACKEND_OVERRIDE` below to
force one instead of auto-selecting.

In [3]:
import shutil

DEFAULT_MODEL = "claude-opus-5"


def has_api_credentials() -> bool:
    return bool(os.environ.get("ANTHROPIC_API_KEY") or os.environ.get("ANTHROPIC_AUTH_TOKEN"))


def claude_code_path() -> str | None:
    return shutil.which("claude")


def codex_path() -> str | None:
    """Find Codex in PATH, an explicit override, or a VS Code installation."""
    override = os.environ.get("MHTDB_CODEX_BINARY", "").strip().strip('"')
    if override:
        p = Path(override).expanduser()
        if p.is_file():
            return str(p.resolve())
        resolved = shutil.which(override)
        if resolved:
            return resolved
    found = shutil.which("codex")
    if found:
        return found
    home_raw = os.environ.get("USERPROFILE") or os.environ.get("HOME")
    if not home_raw:
        return None
    home = Path(home_raw)
    candidates = []
    for extensions in (home / ".vscode" / "extensions", home / ".vscode-insiders" / "extensions"):
        if not extensions.is_dir():
            continue
        for extension in extensions.glob("openai.chatgpt-*"):
            candidates.extend(extension.glob("bin/*/codex.exe"))
    existing = [p for p in candidates if p.is_file()]
    if not existing:
        return None
    try:
        return str(max(existing, key=lambda p: p.stat().st_mtime).resolve())
    except OSError:
        return str(existing[-1].resolve())


def codex_login_status(binary: str | None = None) -> tuple[bool, str]:
    path = binary or codex_path()
    if not path:
        return False, "not installed"
    try:
        proc = subprocess.run([path, "login", "status"], capture_output=True, text=True, timeout=60)
        detail = (proc.stdout or proc.stderr or "unknown status").strip()
        return proc.returncode == 0, detail
    except Exception as exc:
        return False, str(exc)


# Force a choice by setting this to "api" / "claude-code" / "codex" (or leave
# unset / None to auto-select, honouring MHTDB_BACKEND if that env var is set).
BACKEND_OVERRIDE = os.environ.get("MHTDB_BACKEND")
# Force a specific model id; None lets each backend use its own default
# (claude-opus-5 for api/claude-code, the Codex CLI's own configured default).
MODEL_OVERRIDE = None

choice = (BACKEND_OVERRIDE or "auto").lower()
BACKEND = MODEL = why = None

if choice == "api":
    if not has_api_credentials():
        raise SystemExit("BACKEND_OVERRIDE='api' but no ANTHROPIC_API_KEY / ANTHROPIC_AUTH_TOKEN is set")
    BACKEND, why = "api", "ANTHROPIC_API_KEY/ANTHROPIC_AUTH_TOKEN is set"
elif choice in ("claude-code", "cc", "claude_code"):
    cc = claude_code_path()
    if not cc:
        raise SystemExit("BACKEND_OVERRIDE='claude-code' but the `claude` CLI was not found on PATH")
    BACKEND, why = "claude-code", f"forced -- found at {cc}"
elif choice in ("codex", "codex-cli", "codex_cli"):
    path = codex_path()
    if not path:
        raise SystemExit("BACKEND_OVERRIDE='codex' but the Codex CLI was not found")
    logged_in, detail = codex_login_status(path)
    if not logged_in:
        raise SystemExit(f"BACKEND_OVERRIDE='codex' but it is not signed in ({detail})")
    BACKEND, why = "codex", f"forced -- found at {path}, signed in"
elif choice != "auto":
    raise SystemExit(f"unknown BACKEND_OVERRIDE {choice!r}; use api, claude-code, codex, or leave unset")
else:
    cc, cx = claude_code_path(), codex_path()
    if has_api_credentials():
        BACKEND, why = "api", "ANTHROPIC_API_KEY/ANTHROPIC_AUTH_TOKEN is set"
    elif cc:
        BACKEND, why = "claude-code", f"local claude CLI found at {cc}"
    elif cx and codex_login_status(cx)[0]:
        BACKEND, why = "codex", f"local codex CLI found at {cx}, signed in"

RUN_WITH_RULES = BACKEND is None
if RUN_WITH_RULES:
    print("No model backend available (no API key, no local claude/codex CLI signed in).")
    print("Falling back to the offline rule-based extractor: python -m mhtdb.pipeline run --rules")
else:
    MODEL = MODEL_OVERRIDE or (DEFAULT_MODEL if BACKEND != "codex" else None)
    print(f"Using backend : {BACKEND}")
    print(f"Using model   : {MODEL or '(Codex CLI configured default)'}")
    print(f"Why           : {why}")

Using backend : claude-code
Using model   : claude-opus-5
Why           : local claude CLI found at C:\Users\DELL\.local\bin\claude.EXE


### Reset the catalog? (opt-in)

Leave `RESET_CATALOG = False` to add/refresh records for whatever is in
`papers/` on top of any existing catalog. Set it to `True` only when the
paper set has changed and you want the catalog, digitized points, and
review queue to reflect *exactly* the papers currently in `papers/` (this
deletes `catalog/records/`, `catalog/pointers/`, `catalog/points/` first).

In [4]:
RESET_CATALOG = False

if RESET_CATALOG:
    for sub in ("records", "pointers", "points"):
        d = ROOT / "catalog" / sub
        for f in d.glob("*.json"):
            f.unlink()
    (ROOT / "catalog" / "review").mkdir(parents=True, exist_ok=True)
    (ROOT / "catalog" / "review" / "queue.json").write_text("[]", encoding="utf-8")
    print("catalog cleared")
else:
    print("catalog left as-is")

catalog left as-is


### Clean previous outputs

Removes `out/` (the points CSV + boiling-curve plot) and `app/dist/` (the built dashboard) before this run regenerates them, so a stale file from an earlier run -- a plot from a paper set that has since changed, say -- can never be mistaken for this run's result. This does not touch `catalog/`, `pipeline/docmodels/`, `pipeline/figures/`, or `pipeline/cache/`: those are incremental caches that make re-running this notebook free, and wiping them here would force every paper to be re-extracted and re-digitized (real model cost) for no reason. Use `RESET_CATALOG` above if you actually want the catalog itself reset.

In [5]:
# import shutil

# for _d in (ROOT / "out", ROOT / "app" / "dist"):
#     if _d.exists():
#         shutil.rmtree(_d)
#         print(f"removed {_d.relative_to(ROOT)}")
#     else:
#         print(f"{_d.relative_to(ROOT)} not present, nothing to remove")


## S0–S7 — Ingest papers and extract catalog records

For every PDF in `papers/`: parse structure and crop figures (S0), then run
the LLM through triage → numeric conditions → taxonomy → application (S1–S4),
resolve every evidence quote against the source text, normalize units and
compute dimensionless groups (S5), and commit the record (S7).

Pass `--rules` instead to use the offline, keyword-based extractor with no
model calls at all; add `--verify` to run the optional S6 LLM audit pass.

In [6]:
papers = sorted(str(p) for p in (ROOT / "papers").glob("*.pdf"))
if not papers:
    raise SystemExit("No PDFs found in papers/ -- add at least one before running this cell.")
print(f"{len(papers)} paper(s):")
for p in papers:
    print(" ", p)

cmd = [PY, "-m", "mhtdb.pipeline", "run"]
if RUN_WITH_RULES:
    cmd.append("--rules")
else:
    cmd += ["--backend", BACKEND]
    if MODEL:
        cmd += ["--model", MODEL]
run(cmd + papers)

14 paper(s):
  D:\a_claude\AgenticTool-MultiPhaseHUB-main - test11\AgenticTool-MultiPhaseHUB-main\papers\allred_2018_superhydrophobic_trl.pdf
  D:\a_claude\AgenticTool-MultiPhaseHUB-main - test11\AgenticTool-MultiPhaseHUB-main\papers\allred_2019_petal_effect_parahydrophobic.pdf
  D:\a_claude\AgenticTool-MultiPhaseHUB-main - test11\AgenticTool-MultiPhaseHUB-main\papers\berce_2024_wettability_nanoparticle_deposition.pdf
  D:\a_claude\AgenticTool-MultiPhaseHUB-main - test11\AgenticTool-MultiPhaseHUB-main\papers\chu_2013_cuo_hierarchical.pdf
  D:\a_claude\AgenticTool-MultiPhaseHUB-main - test11\AgenticTool-MultiPhaseHUB-main\papers\dharmendra_2026_pool_boiling.pdf
  D:\a_claude\AgenticTool-MultiPhaseHUB-main - test11\AgenticTool-MultiPhaseHUB-main\papers\duan_2020_roughening_techniques.pdf
  D:\a_claude\AgenticTool-MultiPhaseHUB-main - test11\AgenticTool-MultiPhaseHUB-main\papers\hadzic_2022_tio2_nanofluids.pdf
  D:\a_claude\AgenticTool-MultiPhaseHUB-main - test11\AgenticTool-MultiPhaseHUB

backend: claude-code (claude-opus-5)

[allred-2018-superhydrophobic-trl] S0 cached (6p, 1 sections, 6 figures)


  [s1_triage] disk-cached
  S1 -> experimental, dataset=True, pool_boiling
  [s2_conditions] disk-cached
  [s3_taxonomy] disk-cached
  [s4_application] disk-cached


  gating: 1 item(s) queued for review
  reattached existing digitized points (catalog\points\allred-2018-superhydrophobic-trl.points.json)
[allred-2018-superhydrophobic-trl] -> catalog\records\allred-2018-superhydrophobic-trl.json  evidence 21/21 resolved, tags=2, pages=[1, 2, 3, 4, 5]
[allred-2019-petal-effect-parahydrophobic] S0 cached (28p, 21 sections, 21 figures)
  [s1_triage] disk-cached
  S1 -> experimental, dataset=True, pool_boiling
  [s2_conditions] disk-cached
  [s3_taxonomy] disk-cached
  [s4_application] disk-cached


  gating: 1 item(s) queued for review
  reattached existing digitized points (catalog\points\allred-2019-petal-effect-parahydrophobic.points.json)
[allred-2019-petal-effect-parahydrophobic] -> catalog\records\allred-2019-petal-effect-parahydrophobic.json  evidence 24/24 resolved, tags=2, pages=[3, 4, 5, 7, 8, 9, 10, 11, 12, 16, 18]
[berce-2024-wettability-nanoparticle-deposition] S0 cached (17p, 50 sections, 15 figures)
  [s1_triage] disk-cached
  S1 -> experimental, dataset=True, pool_boiling
  [s2_conditions] disk-cached
  [s3_taxonomy] disk-cached
  [s4_application] disk-cached


...

  gating: 2 item(s) queued for review
  reattached existing digitized points (catalog\points\mchale-2011-smooth-sintered-copper.points.json)
[mchale-2011-smooth-sintered-copper] -> catalog\records\mchale-2011-smooth-sintered-copper.json  evidence 22/22 resolved, tags=3, pages=[2, 3, 7, 8, 9, 10, 11, 12, 15, 16, 19, 21, 31, 32]
[moze-2022-laser-textured-sam-untreated-copper] S0 cached (20p, 55 sections, 13 figures)
  [s1_triage] disk-cached
  S1 -> experimental, dataset=True, pool_boiling
  [s2_conditions] disk-cached
  [s3_taxonomy] disk-cached
  [s4_application] disk-cached


  reattached existing digitized points (catalog\points\moze-2022-laser-textured-sam-untreated-copper.points.json)
[moze-2022-laser-textured-sam-untreated-copper] -> catalog\records\moze-2022-laser-textured-sam-untreated-copper.json  evidence 19/19 resolved, tags=2, pages=[1, 4, 5, 6, 8, 9, 10]
[pandey-2024-acoustic-copper-foams] S0 cached (15p, 21 sections, 22 figures)
  [s1_triage] disk-cached
  S1 -> experimental, dataset=True, pool_boiling
  [s2_conditions] disk-cached
  [s3_taxonomy] disk-cached
  [s4_application] disk-cached


  gating: 1 item(s) queued for review
  reattached existing digitized points (catalog\points\pandey-2024-acoustic-copper-foams.points.json)
[pandey-2024-acoustic-copper-foams] -> catalog\records\pandey-2024-acoustic-copper-foams.json  evidence 18/18 resolved, tags=4, pages=[1, 2, 3, 4, 6, 11]
[shi-2015-copper-nanowire-arrays] S0 cached (7p, 5 sections, 13 figures)
  [s1_triage] disk-cached
  S1 -> experimental, dataset=True, pool_boiling
  [s2_conditions] disk-cached
  [s3_taxonomy] disk-cached
  [s4_application] disk-cached


  reattached existing digitized points (catalog\points\shi-2015-copper-nanowire-arrays.points.json)
[shi-2015-copper-nanowire-arrays] -> catalog\records\shi-2015-copper-nanowire-arrays.json  evidence 19/19 resolved, tags=5, pages=[1, 2, 3, 4, 5]

review queue: 5 item(s)   python -m mhtdb.pipeline review
vocab proposals: 2     python -m mhtdb.pipeline propose

14/14 records written to D:\a_claude\AgenticTool-MultiPhaseHUB-main - test11\AgenticTool-MultiPhaseHUB-main\catalog\records
dashboard not rebuilt — run `python app/build.py`, or pass --build


<Popen: returncode: 0 args: ['C:\\Users\\DELL\\AppData\\Local\\Programs\\Pyt...>

## S8 — Crop every figure and table

Crops each detected figure/table into its own PDF (full resolution, vector)
plus a PNG/JPG preview, under `pipeline/figures/<record_id>/`, with a
`crops.json` manifest. This is the input the digitizer (S9) reads from.

In [7]:
run([PY, "-m", "mhtdb.pipeline", "crops"])

$ C:\Users\DELL\AppData\Local\Programs\Python\Python312\python.exe -m mhtdb.pipeline crops


[allred-2018-superhydrophobic-trl] crops cached (4)
[allred-2019-petal-effect-parahydrophobic] crops cached (12)
[berce-2024-wettability-nanoparticle-deposition] crops cached (13)
[chu-2013-cuo-hierarchical] crops cached (5)
[dharmendra-2026-pool-boiling] crops cached (17)
[duan-2020-roughening-techniques] crops cached (17)
[hadzic-2022-tio2-nanofluids] crops cached (10)
[hadzic-2024-inherent-scatter-reference] crops cached (9)
[huang-2023-pure-copper] crops cached (5)
[kim-2016-roughness-moderate-wettability] crops cached (16)
[mchale-2011-smooth-sintered-copper] crops cached (13)
[moze-2022-laser-textured-sam-untreated-copper] crops cached (11)
[pandey-2024-acoustic-copper-foams] crops cached (14)
[shi-2015-copper-nanowire-arrays] crops cached (10)

0 element(s) cropped


<Popen: returncode: 0 args: ['C:\\Users\\DELL\\AppData\\Local\\Programs\\Pyt...>

## S9 — Digitize each paper's one boiling-curve figure

Exactly one figure per paper reaches the real digitizer. A cheap,
tools-disabled, structured-output call (`select_boiling_curve_figure` — the
same fast contract S1–S4 use) sees every figure's caption at once and names
the ONE that is this paper's primary boiling-curve comparison plot — not "is
this A plot" per figure, but "which figure is THE plot" for the whole paper.
A paper with several boiling-curve-shaped figures (a main comparison plus a
zoomed inset, say) still only costs one real extraction; a paper with none
costs only the cheap selection call and is skipped entirely.

Only the winning figure is handed to the agentic `claude` CLI call (tool use
enabled — unlike S1–S4) with a prompt adapted from
`prompts/prompt_pdf-figure_to_data-csv.txt`: it opens the figure itself,
confirms it really is a boiling-curve plot, decides whether it's vector or
raster, calibrates the axes, separates series, and writes its answer back as
JSON. Vector-drawn plots get read straight off the drawing commands;
rasterized plots get pixel-traced against whatever it can calibrate. If it
turns out not to be a plot after all, it
comes back with zero series — not an error; the model is told, explicitly,
that leaving a curve out is better than inventing one it can't support.

Two more speed shortcuts, both free (they reuse what S8 already computed
rather than asking the model to rediscover it):

- **Concurrency across papers** — since it's one figure per paper now,
  `--jobs N` (default 4) runs N papers' selected figures at once instead of
  one at a time. Same total cost, a fraction of the wall-clock time.
- **Skip Step 1 for a known-vector or known-raster crop** — S8's own
  `has_vector`/`has_raster` detection is passed straight into the prompt, so
  the model doesn't spend turns re-probing something already known for free;
  a known-vector crop also gets a smaller turn/effort budget, since reading
  coordinates off paths that already exist needs far less iteration than the
  genuinely open-ended raster case.

In [8]:
run([PY, "-m", "mhtdb.pipeline", "digitize"])

$ C:\Users\DELL\AppData\Local\Programs\Python\Python312\python.exe -m mhtdb.pipeline digitize


[allred-2018-superhydrophobic-trl] already digitized — pass --force to redo, or --record allred-2018-superhydrophobic-trl --figure <id> to redo just one figure
[allred-2019-petal-effect-parahydrophobic] already digitized — pass --force to redo, or --record allred-2019-petal-effect-parahydrophobic --figure <id> to redo just one figure
[berce-2024-wettability-nanoparticle-deposition] already digitized — pass --force to redo, or --record berce-2024-wettability-nanoparticle-deposition --figure <id> to redo just one figure


[chu-2013-cuo-hierarchical] no boiling-curve figure among 5 figure(s) — No caption describes heat flux vs wall superheat curves; Fig 4 shows CHF and HTC comparisons (bar-style), Fig 5 is CHF vs α model.
[dharmendra-2026-pool-boiling] already digitized — pass --force to redo, or --record dharmendra-2026-pool-boiling --figure <id> to redo just one figure
[duan-2020-roughening-techniques] already digitized — pass --force to redo, or --record duan-2020-roughening-techniques --figure <id> to redo just one figure
[hadzic-2022-tio2-nanofluids] already digitized — pass --force to redo, or --record hadzic-2022-tio2-nanofluids --figure <id> to redo just one figure
[hadzic-2024-inherent-scatter-reference] already digitized — pass --force to redo, or --record hadzic-2024-inherent-scatter-reference --figure <id> to redo just one figure
[huang-2023-pure-copper] already digitized — pass --force to redo, or --record huang-2023-pure-copper --figure <id> to redo just one figure
[kim-2016-roughness-moder

<Popen: returncode: 0 args: ['C:\\Users\\DELL\\AppData\\Local\\Programs\\Pyt...>

## S10 — Compile points into a CSV + boiling-curve comparison plot

Selects, per paper, the curve for its plain/reference surface (matched from
the series' legend text or — for a handful of colour-only traced series —
its caption) and overlays it against a CoolProp-based Rohsenow correlation.
Pass `--all-series` to include every traced curve instead of just the plain
references.

A paper whose only boiling-curve figure can't be calibrated (no numeric axis labels in the crop, so the digitizer honestly records normalized 0-1 values instead of guessing real units) contributes no curve here -- reported by record/series with the specific reason, never silently. That is expected on a small or partial catalog, so this stage does not abort the notebook; the CSV is still written and the dashboard build below still runs.

In [9]:
result = run([PY, "-m", "mhtdb.pipeline", "curves", "--out", "out/"], check=False)
if result.returncode != 0:
    print(
        "\nNo boiling-curve plot produced this run (see the reasons printed "
        "above -- most commonly a figure whose axis couldn't be calibrated). "
        "catalog/ and the points CSV are unaffected; continuing to build the "
        "dashboard."
    )

$ C:\Users\DELL\AppData\Local\Programs\Python\Python312\python.exe -m mhtdb.pipeline curves --out out/


2442 point(s) -> out\boiling_points.csv



plotted 7 curve(s) -> out\boiling_curve_summary.png
  allred-2019-petal-effect-parahydrophobic fig-7      raster_digitized_figure  legend matched '\\bsmooth\\b'
  allred-2019-petal-effect-parahydrophobic fig-7      raster_digitized_figure  legend matched '\\bsmooth\\b'
  dharmendra-2026-pool-boiling           fig-8      vector_digitized_figure  legend matched '\\bbare\\b'
  huang-2023-pure-copper                 fig-2      raster_digitized_figure  legend matched '\\bpresent\\s+work\\b'
  mchale-2011-smooth-sintered-copper     fig-3      vector_digitized_figure  legend matched '\\bbare\\b'
  mchale-2011-smooth-sintered-copper     fig-3      vector_digitized_figure  legend matched '\\bbare\\b'
  pandey-2024-acoustic-copper-foams      fig-7      raster_digitized_figure  legend matched '\\bpolished\\b'

68 series excluded (enhanced surfaces, wrong axes, unusable points) — see --all-series to include them

Rohsenow overlay properties: CoolProp water @ 101.3 kPa


## Build the dashboard

Compiles everything in `catalog/` into one self-contained HTML file — no
server, no CORS, opens by double-click.

In [10]:
run([PY, str(ROOT / "app" / "build.py")])

$ C:\Users\DELL\AppData\Local\Programs\Python\Python312\python.exe D:\a_claude\AgenticTool-MultiPhaseHUB-main - test11\AgenticTool-MultiPhaseHUB-main\app\build.py


built D:\a_claude\AgenticTool-MultiPhaseHUB-main - test11\AgenticTool-MultiPhaseHUB-main\app\dist\index.html  (14 records, 290 evidence spans, 14 papers, 5 in review, 221 KB)


<Popen: returncode: 0 args: ['C:\\Users\\DELL\\AppData\\Local\\Programs\\Pyt...>

## Verify the result

In [11]:
from IPython.display import HTML, display

records = sorted((ROOT / "catalog" / "records").glob("*.json"))
print(f"{len(records)} record(s) in catalog/records/")
for r in records:
    rec = json.loads(r.read_text(encoding="utf-8"))
    print(" -", r.stem, "|", rec.get("points_summary", {}).get("n_points", 0), "digitized points")

out_html = ROOT / "app" / "dist" / "index.html"
print(f"\ndashboard: {out_html}  ({out_html.stat().st_size / 1024:.1f} KB)")
display(HTML(f'<a href="{out_html.as_uri()}" target="_blank">Open {out_html.name}</a>'))

14 record(s) in catalog/records/
 - allred-2018-superhydrophobic-trl | 31 digitized points
 - allred-2019-petal-effect-parahydrophobic | 33 digitized points
 - berce-2024-wettability-nanoparticle-deposition | 139 digitized points
 - chu-2013-cuo-hierarchical | 0 digitized points
 - dharmendra-2026-pool-boiling | 95 digitized points
 - duan-2020-roughening-techniques | 156 digitized points
 - hadzic-2022-tio2-nanofluids | 136 digitized points
 - hadzic-2024-inherent-scatter-reference | 164 digitized points
 - huang-2023-pure-copper | 416 digitized points
 - kim-2016-roughness-moderate-wettability | 421 digitized points
 - mchale-2011-smooth-sintered-copper | 214 digitized points
 - moze-2022-laser-textured-sam-untreated-copper | 490 digitized points
 - pandey-2024-acoustic-copper-foams | 61 digitized points
 - shi-2015-copper-nanowire-arrays | 86 digitized points

dashboard: D:\a_claude\AgenticTool-MultiPhaseHUB-main - test11\AgenticTool-MultiPhaseHUB-main\app\dist\index.html  (220.9 KB

In [12]:
from IPython.display import IFrame

# Inline preview (falls back to the link above if your notebook front-end
# blocks local file:// iframes).
IFrame(src=out_html.as_uri(), width="100%", height=700)